# Process data

In [ ]:
import json

with open('bias_flipper_filtered.json','r') as file:
    dataset = json.load(file)

In [ ]:
from collections import defaultdict

grouped = defaultdict(list)
for sample in dataset:
    grouped[sample["story_id"]].append(sample)

data = []
# Traverse the groups
for story_id in sorted(grouped.keys()):
    samples = grouped[story_id]
    for sample in samples:
        if sample['bias'] == 'From the Right':
            title = sample['original_title'] + "\n" if sample['original_title'] else ''
            data.append({
                "story_id": sample["story_id"],
                "title": sample['title'],
                "bias": sample['bias'],
                "text": title + sample['original_body']
            })
            break

In [ ]:
import json

with open('news_sum_center_qwen.json','r') as file:
    data = json.load(file)

In [ ]:
import requests
import time
from tqdm import tqdm

url = ""
MODEL = "Qwen/Qwen3-Embedding-4B"

In [ ]:
import torch

results = []


for sample in tqdm(data):
    user_input = sample['response']

    payload = {
        "model": MODEL,
        "input": user_input,
        "encoding_format": "float"
    }
    headers = {
        "Authorization": "",
        "Content-Type": "application/json"
    }

    response = requests.request("POST", url, json=payload, headers=headers).json()
    pred = torch.tensor(response['data'][0]['embedding'], dtype=torch.float32)
    results.append(pred)

In [ ]:
import pickle

with open("qwen_embeddings.pkl", "wb") as f:
    pickle.dump(results, f)

# News Sum Comparisons

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
import pickle

with open('left_embeddings.pkl', 'rb') as f:
    left_embeddings = pickle.load(f)
    
with open('qwen_embeddings.pkl', 'rb') as f:
    llm_embeddings = pickle.load(f)

In [ ]:
# Embedding dimensions: [64, 128, 256, 512, 768, 1024, 2048]

similarities = []
for left_emb, llm_emb in zip(left_embeddings, llm_embeddings):
    flat1 = F.normalize(left_emb, dim=0)
    flat2 = F.normalize(llm_emb, dim=0)

    cos_sim = torch.dot(flat1, flat2)
    similarities.append(cos_sim)

In [ ]:
with open("qwen_left_sim.pkl", "wb") as f:
    pickle.dump(similarities, f)

# Plot Graph

In [ ]:
import pickle

with open('qwen_left_sim.pkl', 'rb') as f:
    qwen_left_sim = pickle.load(f)
with open('qwen_right_sim.pkl', 'rb') as f:
    qwen_right_sim = pickle.load(f)

with open('ds_left_sim.pkl', 'rb') as f:
    ds_left_sim = pickle.load(f)
with open('ds_right_sim.pkl', 'rb') as f:
    ds_right_sim = pickle.load(f)

with open('gemini_left_sim.pkl', 'rb') as f:
    gemini_left_sim = pickle.load(f)
with open('gemini_right_sim.pkl', 'rb') as f:
    gemini_right_sim = pickle.load(f)
    
with open('gpt_left_sim.pkl', 'rb') as f:
    gpt_left_sim = pickle.load(f)
with open('gpt_right_sim.pkl', 'rb') as f:
    gpt_right_sim = pickle.load(f)

In [ ]:
def process_sim(left, right):
    data = []
    for l, r in zip(left, right):
        data.append([l.item(), r.item()])
    return data

In [ ]:
qwen = process_sim(qwen_left_sim, qwen_right_sim)
ds = process_sim(ds_left_sim, ds_right_sim)
gemini = process_sim(gemini_left_sim, gemini_right_sim)
gpt = process_sim(gpt_left_sim, gpt_right_sim)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

datasets = [np.array(d) for d in [qwen, ds, gemini, gpt]]
labels = 'Qwen', 'Deepseek', 'Gemini', 'GPT'
colors = ['turquoise', 'blue', 'green', 'orange']
markers = ['o', 's', '^', 'D']

fig, ax = plt.subplots(figsize=(8, 6))

for data, color, marker, label in zip(datasets, colors, markers, labels):
    x = data[:, 0]
    y = data[:, 1]
    mean_x, mean_y = x.mean(), y.mean()

    # Plot mean with an edge to stand out
    ax.scatter(mean_x, mean_y, color='none', marker=marker,
               edgecolor=color, linewidth=1.2, s=60, zorder=3, label=f'{label}')

    cov = np.cov(x, y)
    cov += np.eye(2) * 1e-8
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]

    theta = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))

    width, height = 2 * np.sqrt(vals)

    ellipse = Ellipse(xy=(mean_x, mean_y), width=width, height=height,
                      angle=theta, edgecolor=color, facecolor='none', lw=2, zorder=2)
    ax.add_patch(ellipse)

xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
start = max(xmin, ymin)
end = min(xmax, ymax)
grid_color = plt.rcParams['grid.color']
grid_linewidth = plt.rcParams['grid.linewidth']
plt.plot([start, end], [start, end], color=grid_color, linewidth=grid_linewidth)

ax.set_xlabel('left')
ax.set_ylabel('right')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def plot_scatter(group, name, color):
    xs, ys = zip(*group)

    plt.figure(figsize=(4, 4))
    plt.scatter(xs, ys, color=color, s=10)
    
    x_min, x_max = 0, 1
    y_min, y_max = 0, 1

    # Label axes
    plt.xlabel('left')
    plt.ylabel('right')
    plt.title(name)
    
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)

    # Optional: grid, tighter layout
    plt.grid(True)
    grid_color = plt.rcParams['grid.color']
    grid_linewidth = plt.rcParams['grid.linewidth']
    plt.plot([0, 1], [0, 1], color=grid_color, linewidth=grid_linewidth)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_scatter(qwen, 'Qwen', 'turquoise')
plot_scatter(ds, 'Deepseek', 'blue')
plot_scatter(gemini, 'Gemini', 'green')
plot_scatter(gpt, 'GPT', 'orange')